# Video Diagnostics (FFmpeg/FFprobe)

This notebook validates whether a video file may fail MMI video chunking due to keyframe/container issues.

It checks:
- Container/codec metadata
- I-frame (keyframe) count
- Sample keyframe timestamps
- Decode/container errors via ffmpeg

Prerequisites:
- `ffmpeg` and `ffprobe` installed and available in `PATH`

In [2]:
import json
import os
import shutil
import subprocess
from pathlib import Path

# Fallback: local portable ffmpeg/ffprobe downloaded in this repo
LOCAL_FFMPEG_BIN = Path(r"c:\src\mmikustocopilot\.tools\ffmpeg\ffmpeg-8.1-essentials_build\bin")
if LOCAL_FFMPEG_BIN.exists():
    os.environ["PATH"] = str(LOCAL_FFMPEG_BIN) + os.pathsep + os.environ.get("PATH", "")

def run_cmd(cmd):
    """Run a shell command and return (returncode, stdout, stderr)."""
    proc = subprocess.run(cmd, capture_output=True, text=True)
    return proc.returncode, proc.stdout.strip(), proc.stderr.strip()

def require_tools():
    missing = [tool for tool in ("ffprobe", "ffmpeg") if shutil.which(tool) is None]
    if missing:
        raise RuntimeError(f"Missing required tools in PATH: {', '.join(missing)}")

require_tools()
print("ffprobe:", shutil.which("ffprobe"))
print("ffmpeg :", shutil.which("ffmpeg"))

ffprobe: c:\src\mmikustocopilot\.tools\ffmpeg\ffmpeg-8.1-essentials_build\bin\ffprobe.EXE
ffmpeg : c:\src\mmikustocopilot\.tools\ffmpeg\ffmpeg-8.1-essentials_build\bin\ffmpeg.EXE


In [3]:
def diagnose_video(file_path: str, show_keyframes: int = 20):
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")

    print("=" * 60)
    print("VIDEO DIAGNOSTIC REPORT")
    print(f"File: {path.resolve()}")
    print("=" * 60)

    # 1) Container + codec info
    print("\n--- 1. Container & Codec Info ---")
    rc, out, err = run_cmd([
        "ffprobe",
        "-v", "error",
        "-print_format", "json",
        "-show_format",
        "-show_streams",
        str(path),
    ])
    if rc != 0:
        print("ffprobe failed:")
        print(err or out)
        return

    try:
        info = json.loads(out)
    except json.JSONDecodeError:
        info = None

    if info is not None:
        print(json.dumps(info, indent=2)[:5000])
        if len(json.dumps(info)) > 5000:
            print("... (truncated)")
    else:
        print(out[:5000])

    # 2) Keyframe count
    print("\n--- 2. Keyframe Count (I-frames) ---")
    rc, out, err = run_cmd([
        "ffprobe",
        "-v", "error",
        "-select_streams", "v:0",
        "-show_frames",
        "-show_entries", "frame=pict_type",
        "-of", "csv",
        str(path),
    ])
    if rc != 0:
        print("Failed to read frame data:")
        print(err or out)
        return

    iframe_count = sum(1 for line in out.splitlines() if line.endswith(",I") or ",I" in line)
    print(f"I-frame count: {iframe_count}")
    if iframe_count == 0:
        print("RESULT: NO KEYFRAMES FOUND (likely to fail chunking).")
    else:
        print("RESULT: Keyframes detected.")

    # 3) Keyframe timestamps
    print("\n--- 3. Keyframe Timestamps ---")
    rc, out, err = run_cmd([
        "ffprobe",
        "-v", "error",
        "-select_streams", "v:0",
        "-skip_frame", "noref",
        "-show_entries", "frame=pts_time,pict_type",
        "-of", "csv",
        str(path),
    ])
    if rc == 0:
        keyframe_lines = [line for line in out.splitlines() if line.endswith(",I") or ",I" in line]
        if keyframe_lines:
            for line in keyframe_lines[:show_keyframes]:
                print(line)
            if len(keyframe_lines) > show_keyframes:
                print(f"... ({len(keyframe_lines) - show_keyframes} more)")
        else:
            print("No keyframe timestamp rows found.")
    else:
        print("Unable to list keyframe timestamps:")
        print(err or out)

    # 4) Container/decode error check
    print("\n--- 4. Container Error Check (ffmpeg decode) ---")
    rc, out, err = run_cmd([
        "ffmpeg",
        "-v", "error",
        "-i", str(path),
        "-f", "null",
        "-",
    ])

    if not err and not out:
        print("RESULT: No decode/container errors detected.")
    else:
        print("RESULT: Errors detected:")
        print((err or out)[:5000])

    print("\n" + "=" * 60)
    print("SUMMARY")
    print("=" * 60)
    if iframe_count == 0:
        print("ACTION: Re-encode with forced keyframes, for example:")
        print(f"ffmpeg -i \"{path}\" -c:v libx264 -g 30 -keyint_min 15 -sc_threshold 0 output_fixed.mp4")
    else:
        print("Keyframes exist. If request still fails, share full diagnostic output with support.")

In [4]:
# Set your file path and run
VIDEO_FILE = r"C:\Users\jfilcik\OneDrive - Microsoft\Video Project 8 1.mp4"
diagnose_video(VIDEO_FILE)

VIDEO DIAGNOSTIC REPORT
File: C:\Users\jfilcik\OneDrive - Microsoft\Video Project 8 1.mp4

--- 1. Container & Codec Info ---
{
  "streams": [
    {
      "index": 0,
      "codec_name": "h264",
      "codec_long_name": "H.264 / AVC / MPEG-4 AVC / MPEG-4 part 10",
      "profile": "Constrained Baseline",
      "codec_type": "video",
      "codec_tag_string": "avc1",
      "codec_tag": "0x31637661",
      "mime_codec_string": "avc1.424032",
      "width": 1280,
      "height": 720,
      "coded_width": 1280,
      "coded_height": 720,
      "has_b_frames": 0,
      "sample_aspect_ratio": "1:1",
      "display_aspect_ratio": "16:9",
      "pix_fmt": "yuv420p",
      "level": 50,
      "chroma_location": "left",
      "field_order": "progressive",
      "is_avc": "true",
      "nal_length_size": "4",
      "id": "0x1",
      "r_frame_rate": "30/1",
      "avg_frame_rate": "30/1",
      "time_base": "1/15360",
      "start_pts": 0,
      "start_time": "0.000000",
      "duration_ts": 172032